# Non/similarity search
This notebook is shows how to perform similarity search using embeddings and how to use it to find the most dissimilar items.

Let's import the necessary libraries:

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams, RecommendQuery, RecommendInput, RecommendStrategy

load_dotenv("../../../_starter/.env")

True

# Initialize the clients

In [2]:
embedding_client = OpenAI(
    base_url = os.getenv("AZURE_COGNITIVE_ENDPOINT"),
    api_key = os.getenv("AZURE_COGNITIVE_KEY"),
)

qdrant_client = QdrantClient(url="http://localhost:6333")

# Start with basic cosine similarity search
Using the `qdrant_client.search` method, we can perform a similarity search on the collection.

Result is a list of points that are similar to the query vector.

First we will need to convert our search string into a vector.

In [3]:
search_string = "apple"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding
print(query_vector[0:5])

[-0.020793559029698372, 0.014009363017976284, -0.0008607447962276638, 0.01870741881430149, -0.008157994598150253]


Now let's perform a similarity search on the collection from the previous example.

In [4]:
collection_name = "my_random_items"

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=5,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/3287389823.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.9999991, payload={'text': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=4, score=0.46746942, payload={'text': 'pineapple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=1, score=0.46200228, payload={'text': 'banana'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=2, score=0.45880446, payload={'text': 'orange'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=12, version=12, score=0.44105482, payload={'text': 'computer'}, vector=None, shard_key=None, order_value=None)]

As expected, the apple is the most similar item to our query.

Let's try to do it for a different query.

Results should be different now but similar to the new query.

In [5]:
search_string = "computer"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding
print(query_vector[0:5])

[-0.011567637324333191, 0.01772613450884819, -0.008514229208230972, -0.01167961023747921, 0.00329027371481061]


In [6]:
collection_name = "my_random_items"

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=5,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/3287389823.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=12, version=12, score=0.985906, payload={'text': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=0.6211184, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=16, version=16, score=0.60241604, payload={'text': 'keyboard'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=18, version=18, score=0.56685704, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=13, version=13, score=0.54397, payload={'text': 'phone'}, vector=None, shard_key=None, order_value=None)]

# Find dissimilar items
Let's flips this on its head and find the most dissimilar items now.

First lets generate the new vector first.

In [7]:
search_string = "banana"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding
print(query_vector[0:5])

[-0.007021163124591112, -0.00937414076179266, -0.006371545139700174, 0.026437947526574135, -0.030305441468954086]


We can get the all 20 points from the collection and keep only the last 5.

In [8]:
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=20,
)

reversed_results = results[-5:]
reversed_results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/4115420862.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=19, version=19, score=0.25592452, payload={'text': 'chair'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=8, score=0.2523036, payload={'text': 'house'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=0.2510685, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=17, score=0.23831232, payload={'text': 'monitor'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=18, version=18, score=0.23101115, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None)]

This give us pretty good results, as a chair is not very similar to a banana.

However this oparation might be costly and time consuming on larger datasets.

# Experimentation

This is where I encourage you to experiment with the data, vectors and queries on your own.

For example we can try to reverse each element of the vector and try to search again.

This is very naive approach and might not work well but let's try it anyway.

In [9]:
reverse_query_vector = [-x for x in query_vector]

print(reverse_query_vector[0:5])

[0.007021163124591112, 0.00937414076179266, 0.006371545139700174, -0.026437947526574135, 0.030305441468954086]


Let's try to find the most simillar items to this new vector.

In [10]:
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=reverse_query_vector,
    limit=5,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/340927645.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=18, version=18, score=-0.23101115, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=17, score=-0.23831232, payload={'text': 'monitor'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=-0.2510685, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=8, score=-0.2523036, payload={'text': 'house'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=19, version=19, score=-0.25592452, payload={'text': 'chair'}, vector=None, shard_key=None, order_value=None)]

Results are decent, but not perfect, try to find a better way to do this.